# SD image pilot on Kaggle

- **Created**: 2026-11-09T13:24:35+07:00
- **Last Updated**: 2026-11-09T15:00:00+07:00
- **Author**: Group 4
- **Objective**: Run pretrained T2I, img2img and native inpainting on real Drive inputs. No training, detector claims or silent mock fallback.
- **Prerequisites**: Kaggle GPU, Internet ON. Upload the provided SD_IMAGE_DEMO_KAGGLE.zip as a private Dataset and attach it here. Do not install or replace Kaggle's torch/torchvision.

## References & Consumed Artifacts
- [Source pipeline](../src/pipelines/sd_image_demo.py)
- [Config](../configs/generation/kaggle_image_demo.yaml)
- [Guide and limitations](../docs/reports/KAGGLE_SD_IMAGE_DEMO.md)
- [Generated artifacts](../experiments/runs/)
- [Notebook rule](../.agents/rules/NOTEBOOK_HEADER_CONVENTION.md)


## Roadmap

| Step | Description | What it does | Import path |
| --- | --- | --- | --- |
| 1 | Environment | Extract code bundle, install generation extras | requirements/generation.txt |
| 2 | Inputs | Download only 3 adult FFHQ crops and inspect masks | src.pipelines.prepare_image_demo |
| 3 | Inference | Run 3 native pretrained pipelines | src.pipelines.sd_image_demo |
| 4 | Inspection | View output gallery and download artifacts | src.common.visualizer |

Quick links: [Engine](../src/models/diffusion/stable_diffusion_engine.py), [Input loader](../src/data/sd_demo_data.py), [Preview](../src/common/visualizer.py), [Generation](../src/pipelines/generate_deepfake_images.py).


In [ ]:
from pathlib import Path
import os
import sys
import zipfile

workspace = Path.cwd()
if (workspace / "src/pipelines/sd_image_demo.py").is_file():
    repo = workspace
elif (workspace / "uth-cv-coursework/src/pipelines/sd_image_demo.py").is_file():
    repo = workspace / "uth-cv-coursework"
else:
    import shutil
    input_dir = workspace.parent / "input"
    archives = sorted(input_dir.rglob("SD_IMAGE_DEMO_KAGGLE.zip"))
    unpacked = [p.parents[2] for p in input_dir.rglob("sd_image_demo.py")
                if p.parent.name == "pipelines" and p.parent.parent.name == "src"
                and (p.parents[2] / "configs/generation/kaggle_image_demo.yaml").is_file()]
    repo = workspace / "uth-cv-coursework"
    if unpacked:
        shutil.copytree(unpacked[0], repo, dirs_exist_ok=True)
    elif archives:
        with zipfile.ZipFile(archives[0]) as bundle:
            for member in bundle.infolist():
                if not (repo / member.filename).resolve().is_relative_to(repo.resolve()):
                    raise ValueError("Unsafe archive path")
            bundle.extractall(repo)
    else:
        raise FileNotFoundError("Attach SD_IMAGE_DEMO_KAGGLE.zip or its extracted code Dataset first.")
os.chdir(repo)
sys.path.insert(0, str(repo))
repo


Inference dependencies are installed inside worker venvs. The notebook kernel installs only requirements/notebook.txt. If an older cell already downgraded Hub/protobuf globally, save results and start a fresh Kaggle session before using this notebook; restarting only the Python kernel does not undo pip changes.

In [ ]:
%pip install -q -r requirements/notebook.txt


In [ ]:
import torch
import yaml
from IPython.display import HTML, Image, display, FileLink
from src.pipelines.prepare_image_demo import prepare_drive_demo
from src.pipelines.sd_demo_launcher import run_isolated_demo as run_demo
from src.common.visualizer import save_sd_input_preview

assert torch.cuda.is_available(), "Enable GPU in Kaggle notebook settings."
config_path = Path("configs/generation/kaggle_image_demo.yaml")
config = yaml.safe_load(config_path.read_text())
display({"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__})


## Prepare and inspect real inputs
Only three matching adult FFHQ crops are downloaded. The public folder listing is not a bulk downloader. If Drive is unavailable, attach the original data to Kaggle and point the config at a local JSONL manifest.


In [ ]:
data = config["data"]
selected = prepare_drive_demo(data["drive_folder_id"], Path(data["input_manifest"]), Path(data["download_root"]), limit=data["limit"])
display([{k: r[k] for k in ("image_id", "file_path", "split")} for r in selected])
preview = Path(config["run_dir"]) / "input_preview.png"
generation = config["generation"]
save_sd_input_preview(Path(data["input_manifest"]), preview, (generation["width"], generation["height"]), tuple(generation["mask_box_xyxy_normalized"]))
display(Image(filename=str(preview)))


## Check the masks before continuing
White marks the edit region. The default ellipse is only a pilot mouth ROI on aligned crops, not a landmark detector. If misplaced, add a mask_path per source in the JSONL or adjust the normalized xyxy box in the config. Native img2img does not guarantee identity/pose preservation. T2I uses text only.


In [ ]:
summary = run_demo(config_path)
display(summary)


## Inspect outputs
This is technical QA plus visual inspection. Landmark/CLIP curation and dinoViT evaluation are not claimed. A low sharpness value is not by itself evidence of a deepfake, and no detector score is fabricated.


In [ ]:
run_dir = Path(config["run_dir"])
import html
gallery = html.escape((run_dir / "gallery.html").read_text(encoding="utf-8"), quote=True)
display(HTML(f'<iframe title="Generated images" srcdoc="{gallery}" style="width:100%;height:850px;border:0"></iframe>'))
display(FileLink(str(run_dir / "generated_manifest.jsonl")))
display(FileLink(str(run_dir / "summary.json")))


In [ ]:
import shutil
archive = shutil.make_archive(str(run_dir.parent / "sd_image_demo_outputs"), "zip", root_dir=run_dir)
display(FileLink(archive))
